<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2019%20-%20MAKEUP%20FRI%20OCT%2030%20-%20FINAL%20PREP/final_prep_himalaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Final Practice: Planning a Himalayan Expedition**

You work for an agency that runs guided expeditions in the Nepal Himalaya. Before each season, operations has two planning questions:

1. **How many oxygen sets should we bring for an expedition?** A missing oxygen set high on the mountain costs the agency **10 times** what carrying a spare one does.
2. **Which climbers are at high risk of dying?**

**It is the end of 2014.** You build your models on the seasons 2000–2014, then see how they do on the next five (2015–2019).

**Practice, not graded.** Open this notebook in Colab (button above) and use **Gemini in Colab** for the code: click the Gemini (✦) button at the top right, or "Generate" in an empty code cell. Compare your results with your classmates': this is a friendly competition. Write the one-sentence answers yourself. On the Final you read output, you don't run code.

| Part | Topic | Time |
| --- | --- | --- |
| 1 | Know your data | ~20 min |
| 2 | Modeling and evaluation | ~20 min |
| 3 | Interpretation and drilling deeper | ~20 min |

**Data:** The Himalayan Database via TidyTuesday (2020-09-22): expeditions and climbers on Nepal's peaks, 1905–2019.

## Setup *(just run these cells)*

In [ ]:
# Load the three Himalayan Database files
import pandas as pd

url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-09-22/'
members = pd.read_csv(url + 'members.csv')
exps = pd.read_csv(url + 'expeditions.csv')
peaks = pd.read_csv(url + 'peaks.csv')

In [ ]:
# Expedition table: one row per expedition, ending with the number of oxygen sets its climbers used.
# Team size and hired staff are counted from the same climber list as the oxygen sets, so they always match.
# (One expedition ID is used twice, so expeditions are matched on ID and year.)
key = ['expedition_id', 'year']
team = members.groupby(key).agg(team_size=('member_id', 'count'),
                                hired_staff=('hired', 'sum'),
                                o2_sets=('oxygen_used', 'sum')).reset_index()
peak_height = peaks[['peak_id', 'height_metres']].rename(columns={'height_metres': 'peak_height_metres'})

expeditions = exps[['expedition_id', 'peak_id', 'year', 'season', 'basecamp_date', 'termination_date',
                    'oxygen_used', 'termination_reason']]
expeditions = expeditions.merge(peak_height, on='peak_id', how='left').merge(team, on=key)
expeditions = expeditions[['expedition_id', 'peak_id', 'year', 'season', 'basecamp_date', 'termination_date',
                           'peak_height_metres', 'team_size', 'hired_staff', 'oxygen_used', 'termination_reason',
                           'o2_sets']]
expeditions['basecamp_date'] = pd.to_datetime(expeditions['basecamp_date'])
expeditions['termination_date'] = pd.to_datetime(expeditions['termination_date'])
expeditions

In [ ]:
# Climber table: one row per climber, ending with whether they died
climbers = members.merge(peak_height, on='peak_id', how='left')
climbers = climbers[['member_id', 'peak_id', 'year', 'season', 'sex', 'age', 'citizenship', 'hired',
                     'peak_height_metres', 'highpoint_metres', 'death_cause', 'died']]
climbers

In [ ]:
# Train on the seasons 2000-2014; test on the next five (2015-2019)
TRAIN_START_YEAR = 2000
TEST_START_YEAR = 2015

### What each column means

**`expeditions`** — one row per expedition

| Column | Meaning |
| --- | --- |
| `expedition_id` | Code for the expedition |
| `peak_id` | Code for the peak (for example `EVER` = Everest) |
| `year`, `season` | When the expedition took place |
| `basecamp_date` | Date the team reached base camp |
| `termination_date` | Date the expedition ended |
| `peak_height_metres` | Height of the peak |
| `team_size` | Number of climbers listed on the expedition, hired staff included |
| `hired_staff` | Number of listed climbers who were hired staff (for example Sherpa guides) |
| `oxygen_used` | Whether anyone on the expedition used bottled oxygen |
| `termination_reason` | Why the expedition ended (reached the summit, bad weather, an accident, ...) |
| `o2_sets` | **Target for Question 3:** number of listed climbers who used bottled oxygen |

**`climbers`** — one row per climber on an expedition

| Column | Meaning |
| --- | --- |
| `member_id` | Code for the climber on that expedition |
| `peak_id`, `year`, `season`, `peak_height_metres` | Same as in `expeditions` |
| `sex`, `age`, `citizenship` | The climber |
| `hired` | Whether the climber was hired staff rather than a paying client |
| `highpoint_metres` | Highest point the climber reached |
| `death_cause` | Cause of death, for climbers who died |
| `died` | **Target for Question 4:** whether the climber died on the expedition |

# **Part 1: Know your data** (~20 min)

Missing values, duplicates and distinct values, already counted for you:

In [ ]:
# For each table: rows, exact duplicate rows, and per column the type, missing values and distinct values
for name, table in [('expeditions', expeditions), ('climbers', climbers)]:
    print(f'{name}: {len(table)} rows, {table.duplicated().sum()} exact duplicate rows')
    display(pd.DataFrame({'type': table.dtypes, 'missing': table.isna().sum(), 'distinct': table.nunique()}))

In [ ]:
# Remove exact duplicate rows
expeditions = expeditions.drop_duplicates()
climbers = climbers.drop_duplicates()
print(len(expeditions), 'expeditions and', len(climbers), 'climbers left')

### Question 1

Prepare both tables for modeling.

**1a.** Which columns will you remove before modeling, and why?

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

**1b.** What will you do with the missing values?

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

**1c. Custom cell.** Any other preparation you decide on (for example, turning text columns into dummy variables).

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

### Question 2

Which climbers and expeditions does this data describe well, and which hardly at all? Figures could help.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# **Part 2: Modeling and evaluation** (~20 min)

A tool for scoring your oxygen plans, ready to use:

In [ ]:
# Tool for scoring your oxygen plans (just run this cell)
# Question 3 - the cost of an oxygen plan for each expedition:
#   brought: sets the agency brings; actual: sets the climbers needed
#   - each spare set (brought but not needed) costs 1: money and porter weight
#   - each missing set (needed but not brought) costs missing_cost: a climber goes without oxygen high up.
#     The agency says that is 10 times worse than a spare set, so missing_cost=10.
def oxygen_cost(brought, actual, missing_cost=10):
    brought_cost = 1 # Relative to the missing cost.
    missing = (actual - brought).clip(lower=0)   # sets that were needed but not brought
    spare = (brought - actual).clip(lower=0)     # sets carried for nothing
    cost = missing_cost * missing.sum() + brought_cost * spare.sum()
    print(f'Short on {(missing > 0).mean():.0%} of expeditions, {spare.mean():.1f} spare sets each, total cost {cost:,.0f}')
    return cost

### Question 3

How many oxygen sets should the agency bring on each 2015–2019 expedition? Can you beat what it would do with no model, scored with the `oxygen_cost` function above?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 4

Predict which climbers on 2015–2019 expeditions will die (`died`). Decide where to draw the line between "high risk" and not, and defend that choice to the agency.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Leaderboard

Post your **total oxygen cost** on 2015–2019 (Question 3), and your death model's **share of deaths caught** together with the **share of flagged climbers who died** (Question 4).

# **Part 3: Interpretation and drilling deeper** (~20 min)

### Question 5

Which inputs matter most to your best model for each question, and how can you tell?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 6

Does your death model work equally well for women and men? Find out why.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 7

Does your model hold up year by year after 2014? Explain what you see.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*